# Hooks — Jane Street, February 2014

[https://www.janestreet.com/puzzles/hooks-index/](https://www.janestreet.com/puzzles/hooks-index/)

## Goals

The goal of this notebook is to use a solver to solve Hooks. I am trying to learn to use CP-SAT.
The puzzle itself is fairly straightforward to solve by hand and has a simple rule set, making it
a great candidate for me to practice!

## AI use disclaimer

I used AI to help me with Python syntax and debugging, but all code and text is fully mine. The
goal of this is to learn, and the solution is already available online.

## Rules

An N×N grid, with a hook defined as the region where `max(row, column) = hook_value`. Using
zero-based indexing, hook 1 is just the single square `(0, 0)`, and hook 9 is the full bottom row
plus the rightmost column.

Each hook contains exactly *n* instances of its own hook value — hook 8 contains eight 8s, hook 9
contains nine 9s. The rest of the grid is filled with 0s, and the rows and columns must sum to the
values given along the border.

In [55]:
import math
from ortools.sat.python import cp_model

## Setup, clues, and helper functions

We define constants for the clues and the grid size, then some helper functions.

`sum_hook` in particular will be useful for the constraints, since a neat way to encode the
"nine 9s, eight 8s, ..." rule is to require `sum_hook(n) == n²`. Because every cell in hook *n*
is either 0 or *n*, a hook summing to *n²* means exactly *n* of its cells are filled.

In [56]:
ROW_SUMS = [26, 42, 11, 22, 42, 36, 29, 32, 45]
COL_SUMS = [31, 19, 45, 16, 5, 47, 28, 49, 45]
N = 9

In [57]:
def sum_row(grid, i):
    return sum(grid[i])


def sum_col(grid, j):
    return sum(grid[i][j] for i in range(N))


def sum_hook(grid, hook_value):
    return sum(
        grid[i][j] for i in range(N) for j in range(N) if max(i, j) == hook_value - 1
    )


def show(grid):
    print(" ".join(str(COL_SUMS[i]) for i in range(N)))
    print(" ".join("-" * N))
    for i in range(N):
        print(
            str(ROW_SUMS[i])
            + "| "
            + " ".join(str(grid[i][j]) for j in range(N))
            + " | = "
            + str(sum_row(grid, i))
        )
    print(" ".join("-" * 2 * N))
    print(" ".join(str(sum_col(grid, j)) for j in range(N)))

`show` prints a grid with its border clues, plus the actual row and column totals so they can be
compared against the clues at a glance. Below is a dummy grid used to test it.

In [58]:
EXAMPLE = [
    [1, 2, 3, 4, 5, 6, 7, 8, 9],
    [2, 3, 4, 5, 6, 7, 8, 9, 1],
    [3, 4, 5, 6, 7, 8, 9, 1, 2],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [5, 6, 7, 8, 9, 1, 2, 3, 4],
    [6, 7, 8, 9, 1, 2, 3, 4, 5],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [8, 9, 1, 2, 3, 4, 5, 6, 7],
    [9, 9, 9, 9, 9, 9, 9, 9, 9],
]

show(EXAMPLE)

31 19 45 16 5 47 28 49 45
- - - - - - - - -
26| 1 2 3 4 5 6 7 8 9 | = 45
42| 2 3 4 5 6 7 8 9 1 | = 45
11| 3 4 5 6 7 8 9 1 2 | = 45
22| 4 5 6 7 8 9 1 2 3 | = 45
42| 5 6 7 8 9 1 2 3 4 | = 45
36| 6 7 8 9 1 2 3 4 5 | = 45
29| 7 8 9 1 2 3 4 5 6 | = 45
32| 8 9 1 2 3 4 5 6 7 | = 45
45| 9 9 9 9 9 9 9 9 9 | = 81
- - - - - - - - - - - - - - - - - -
45 53 52 51 50 49 48 47 46


## Solution checker

We write a checker that enforces the rules of the puzzle.  Our example should fail the checker because it breaks the hooks constraint.

In [59]:
def check(grid, ROW_SUMS, COL_SUMS):
    for i in range(N):
        if sum_row(grid, i) != ROW_SUMS[i]:
            return False
    for j in range(N):
        if sum_col(grid, j) != COL_SUMS[j]:
            return False
    for hook_value in range(1, N + 1):
        if sum_hook(grid, hook_value) != hook_value**2:
            return False
    return True


check(EXAMPLE, ROW_SUMS=[45] * 8 + [81], COL_SUMS=[45, 53, 52, 51, 50, 49, 48, 47, 46])

False

## Specify the model

Now we add the variables the model will solve for. We need one variable per cell, and the domain
of each is exactly `{0, hook_value}`, where `hook_value` is whichever hook that cell belongs to.

Notice that we restrict each domain as tightly as possible — so in a sense we are already encoding
part of the puzzle in the variable definitions rather than in the constraints.

In [60]:
def new_grid_variables(model):
    grid = []
    for i in range(N):
        row = []
        for j in range(N):
            hook_value = max(i, j) + 1
            variable = model.new_int_var_from_domain(
                cp_model.Domain.from_values([0, hook_value]), f"grid_{i}_{j}"
            )
            row.append(variable)
        grid.append(row)
    return grid

Now, we will add constraints to the model according to the ruleset.

1.  Rows must sum to ROW_SUMS, cols must sum to COL_SUMS.  We can enforce this by adding the linear constraints.

2. We must have n instances of n in the nth hook.  We can enforce this by setting hook_sum_n = n^2 for n in 1:9.  

In [63]:
def build_base():
    model = cp_model.CpModel()
    cells = new_grid_variables(model)
    for i, val in enumerate(ROW_SUMS):
        model.add(val == sum_row(cells, i))
    for j, val in enumerate(COL_SUMS):
        model.add(val == sum_col(cells, j))
    for hook_value in range(1, N + 1):
        model.add(hook_value**2 == sum_hook(cells, hook_value))
    return model, cells

In [64]:
model, cells = build_base()
print(model.model_stats())

satisfaction model '': (model_fingerprint: 0x1bb7e75bd74a3559)
#Variables: 81 (56 primary variables)
  - 1 Booleans in [0,1]
  - 3 in [0][2]
  - 5 in [0][3]
  - 7 in [0][4]
  - 9 in [0][5]
  - 11 in [0][6]
  - 13 in [0][7]
  - 15 in [0][8]
  - 17 in [0][9]
#kLinear1: 1
#kLinear3: 1
#kLinearN: 25 (#terms: 239)


The model now has 81 variables with the right number in each hook, and 27 constraints: 9 for the
row sums, 9 for the column sums, and 9 for the hook sums.

## Add grid extraction and status function

A couple helper functions to extract the solved grid and print us stats about our solve.

In [ ]:
def read_grid_from_solver(solver, cells):
    grid = []
    for i in range(N):
        row = []
        for j in range(N):
            row.append(solver.value(cells[i][j]))
        grid.append(row)
    return grid


def report(solver, status, label=""):
    """Print the statistics worth looking at after every solve."""
    print(label)
    print(f"  status     : {solver.status_name(status)}")
    print(f"  branches   : {solver.num_branches}")
    print(f"  conflicts  : {solver.num_conflicts}")
    print(f"  wall time  : {solver.wall_time:.4f}s")

## The solution

Here, we just build the model and run the solver on it.  Since the problem is so constrained, we don't need to add any additional information to prune the search space.

In [ ]:
model, cells = build_base()

solver = cp_model.CpSolver()
status = solver.solve(model)
report(solver, status)

grid = read_grid_from_solver(solver, cells)
show(grid)
print(grid)


  status     : OPTIMAL
  branches   : 0
  conflicts  : 0
  wall time  : 0.0034s
31 19 45 16 5 47 28 49 45
- - - - - - - - -
26| 1 0 3 0 0 6 7 0 9 | = 26
42| 2 2 3 0 5 6 7 8 9 | = 42
11| 3 0 0 0 0 0 0 8 0 | = 11
22| 4 4 4 4 0 6 0 0 0 | = 22
42| 5 5 5 5 0 6 7 0 9 | = 42
36| 0 0 6 0 0 6 7 8 9 | = 36
29| 7 0 7 7 0 0 0 8 0 | = 29
32| 0 8 8 0 0 8 0 8 0 | = 32
45| 9 0 9 0 0 9 0 9 9 | = 45
- - - - - - - - - - - - - - - - - -
31 19 45 16 5 47 28 49 45
[[1, 0, 3, 0, 0, 6, 7, 0, 9], [2, 2, 3, 0, 5, 6, 7, 8, 9], [3, 0, 0, 0, 0, 0, 0, 8, 0], [4, 4, 4, 4, 0, 6, 0, 0, 0], [5, 5, 5, 5, 0, 6, 7, 0, 9], [0, 0, 6, 0, 0, 6, 7, 8, 9], [7, 0, 7, 7, 0, 0, 0, 8, 0], [0, 8, 8, 0, 0, 8, 0, 8, 0], [9, 0, 9, 0, 0, 9, 0, 9, 9]]
True


## Check the solution

We run our checker over the grid the solver returned, rather than trusting it.

In [69]:
print(check(grid, ROW_SUMS, COL_SUMS))

True


The solution is valid. The final step is to extract the answer: the sum of the values in the
shaded squares. The shading in the puzzle image is a checkerboard with `(0, 0)` shaded, so the
shaded cells are those where `row + column` is even.

In [71]:
sum = 0
for i in range(N):
    for j in range(N):
        if (i + j) % 2 == 0:
            sum += grid[i][j]
print(sum)

158


The correct solution is 158.